In [30]:
import requests
import json
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv

load_dotenv('secrets.env')

True

In [31]:
# Google Authentication
PROJECT_ID = os.getenv('PROJECT_ID')
DATASET_ID = os.getenv('DATASET_ID')
TABLE_ID = 'vend_customers'

# Replace with the path to your service account key file
SERVICE_ACCOUNT_FILE = os.getenv('SERVICE_ACCOUNT_FILE')


In [33]:
# Initialize BigQuery client
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

# Define the BigQuery table reference
dataset_ref = client.dataset(DATASET_ID)
table_ref = dataset_ref.table(TABLE_ID)

In [35]:

schema = [
    bigquery.SchemaField("name", "STRING"),
    bigquery.SchemaField("id", "STRING"),
    bigquery.SchemaField("customer_code", "STRING"),
    bigquery.SchemaField("source_unique_id", "STRING"),
    bigquery.SchemaField("first_name", "STRING"),
    bigquery.SchemaField("last_name", "STRING"),
    bigquery.SchemaField("email", "STRING"),
    bigquery.SchemaField("year_to_date", "FLOAT"),
    bigquery.SchemaField("balance", "FLOAT"),
    bigquery.SchemaField("loyalty_balance", "FLOAT"),
    bigquery.SchemaField("on_account_limit", "FLOAT"),
    bigquery.SchemaField("note", "STRING"),
    bigquery.SchemaField("gender", "STRING"),
    bigquery.SchemaField("date_of_birth", "DATE"),
    bigquery.SchemaField("company_name", "STRING"),
    bigquery.SchemaField("do_not_email", "BOOLEAN"),
    bigquery.SchemaField("loyalty_email_sent", "BOOLEAN"),
    bigquery.SchemaField("phone", "STRING"),
    bigquery.SchemaField("mobile", "STRING"),
    bigquery.SchemaField("fax", "STRING"),
    bigquery.SchemaField("twitter", "STRING"),
    bigquery.SchemaField("website", "STRING"),
    bigquery.SchemaField("physical_address_1", "STRING"),
    bigquery.SchemaField("physical_address_2", "STRING"),
    bigquery.SchemaField("physical_suburb", "STRING"),
    bigquery.SchemaField("physical_city", "STRING"),
    bigquery.SchemaField("physical_postcode", "STRING"),
    bigquery.SchemaField("physical_state", "STRING"),
    bigquery.SchemaField("physical_country_id", "STRING"),
    bigquery.SchemaField("postal_address_1", "STRING"),
    bigquery.SchemaField("postal_address_2", "STRING"),
    bigquery.SchemaField("postal_suburb", "STRING"),
    bigquery.SchemaField("postal_city", "STRING"),
    bigquery.SchemaField("postal_state", "STRING"),
    bigquery.SchemaField("postal_country_id", "STRING"),
    bigquery.SchemaField("customer_group_id", "STRING"),
    bigquery.SchemaField("enable_loyalty", "BOOLEAN"),
    bigquery.SchemaField("custom_field_1", "STRING"),
    bigquery.SchemaField("custom_field_2", "STRING"),
    bigquery.SchemaField("custom_field_3", "STRING"),
    bigquery.SchemaField("custom_field_4", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
    bigquery.SchemaField("updated_at", "TIMESTAMP"),
    bigquery.SchemaField("deleted_at", "TIMESTAMP"),
    bigquery.SchemaField("customer_group_ids", "STRING", mode="REPEATED"),
    bigquery.SchemaField("version", "INTEGER"),
    bigquery.SchemaField("postal_postcode", "STRING"),
    bigquery.SchemaField("time_until_deletion", "STRING")
]

In [36]:
# Create the table if it doesn't exist
table = bigquery.Table(table_ref, schema=schema)
table = client.create_table(table, exists_ok=True)

In [37]:
# Function to fetch data from Vend API
def fetch_vend_data(url, headers, params=None):
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()


In [38]:
# Function to load data into BigQuery
def load_data_to_bigquery(client, table_ref, rows_to_insert):
    errors = client.insert_rows_json(table_ref, rows_to_insert)
    if errors:
        print(f"Encountered errors while inserting rows: {errors}")
    else:
        print("Data successfully inserted into BigQuery.")


In [39]:
# Vend API details
vend_url = "https://ashcorp.retail.lightspeed.app/api/2.0/customers"
vend_headers = {
    "accept": "application/json",
    "authorization": f"Bearer {os.getenv('LIGHTSPEED_ACCESS_TOKEN')}"
}

In [45]:
 #Verify table existence
try:
    client.get_table(table_ref)
    print(f"Table {TABLE_ID} found in dataset {DATASET_ID}.")
except Exception as e:
    print(f"Error: {e}")
    print(f"Table {TABLE_ID} not found in dataset {DATASET_ID}. Please check the table name and dataset.")
    exit(1)

# Function to insert rows with retry mechanism
def insert_rows_with_retry(client, table_ref, rows, max_retries=3):
    for attempt in range(max_retries):
        try:
            errors = client.insert_rows_json(table_ref, rows)
            if not errors:
                print("Data inserted successfully.")
                return
            else:
                print(f"Errors occurred while inserting rows: {errors}")
        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            time.sleep(2 ** attempt)  # Exponential backoff
    print("Max retries reached. Failed to insert rows.")

Table vend_customers found in dataset vend_raw.


In [ ]:
# Pagination parameters
after = 0

# Extract, Transform, Load (ETL) process
while True:
    # Extract data from Vend API
    params = {"after": after}
    data = fetch_vend_data(vend_url, vend_headers, params)

    # Transform data (if needed)
    rows_to_insert = data["data"]

    # Load data into BigQuery with retry mechanism
    insert_rows_with_retry(client, table_ref, rows_to_insert)
    
    # Get the max version number for the next request
    after = data["version"]["max"]

    # Check if the data collection is empty
    if not data["data"]:
        break

print("ETL process completed successfully.")